## Inhalte aus dem XML extrahieren:

In [1]:
import requests
from bs4 import BeautifulSoup as soup
import unicodedata
from lxml import etree
import pandas as pd

#### Datenset aus Datei laden:

In [2]:
tree = etree.parse('datadump.xml')

root = tree.getroot()                
ns = {'marc': 'http://www.loc.gov/MARC21/slim'} 
records = root.findall('.//marc:record', namespaces=ns)
print("Gefundene Records:", len(records))

Gefundene Records: 706


Einblick:

In [3]:
testrecord = records[54]
print(testrecord)

<Element {http://www.loc.gov/MARC21/slim}record at 0x72eb99dab580>


In [4]:
testrecord_text = etree.tostring(testrecord, encoding="unicode")
print(testrecord_text)

<record xmlns="http://www.loc.gov/MARC21/slim" type="Bibliographic">
<leader>00000nam a2200000uc 4500</leader>
<controlfield tag="001">1341143651</controlfield>
<controlfield tag="003">DE-101</controlfield>
<controlfield tag="005">20250321171048.0</controlfield>
<controlfield tag="007">cr||||||||||||</controlfield>
<controlfield tag="008">240905s2024    gw |||||o|||| 00||||ger  </controlfield>
<datafield ind1=" " ind2=" " tag="015">
<subfield code="a">24,O10</subfield>
<subfield code="2">dnb</subfield>
</datafield>
<datafield ind1="7" ind2=" " tag="016">
<subfield code="2">DE-101</subfield>
<subfield code="a">1341143651</subfield>
</datafield>
<datafield ind1=" " ind2=" " tag="020">
<subfield code="a">9783384339881</subfield>
<subfield code="9">978-3-384-33988-1</subfield>
</datafield>
<datafield ind1=" " ind2=" " tag="020">
<subfield code="a">3384339886</subfield>
<subfield code="9">3-384-33988-6</subfield>
</datafield>
<datafield ind1="3" ind2=" " tag="024">
<subfield code="a">978338

In [5]:
# Extrahieren des Inhalts des Controlfields "001"
controlfield_001 = testrecord.find(".//controlfield[@tag='001']")
print(controlfield_001)

None


In [6]:
# Definieren des Namespaces: 
namespace = {'marc': 'http://www.loc.gov/MARC21/slim'}

In [7]:
controlfield_001 = testrecord.find('.//marc:controlfield[@tag="001"]', namespaces=namespace)#.text
print(controlfield_001)

<Element {http://www.loc.gov/MARC21/slim}controlfield at 0x72eb99dbaec0>


In [8]:
datafield_245_a = testrecord.xpath('.//marc:datafield[@tag="245"]/marc:subfield[@code="a"]/text()', namespaces=namespace)
print(datafield_245_a)

['Jerusalem im Zeichen des Kreuzes']


In [9]:
print(type(datafield_245_a))

<class 'list'>


### Erstellen einer Funktion, um Inhalte nach Feldern aus XML zu extrahieren:

In [14]:
def parse_record(record):
    
    record = etree.tostring(record, encoding="unicode").replace('\x98', '').replace('\x9c', '')
    #record = str(record).replace('\x98', '').replace('\x9c', '')
    ns = {"marc": "http://www.loc.gov/MARC21/slim"}
    xml = etree.fromstring(unicodedata.normalize("NFC", record))
    
       
    def extract_text(xpath_query):
        fields = xml.xpath(xpath_query, namespaces=ns)
        if fields:
            return "; ".join(field.text for field in fields if field.text)
        return None

    idn = extract_text("marc:controlfield[@tag='001']")
    titel = extract_text("marc:datafield[@tag='245']/marc:subfield[@code='a']")
    author = extract_text("marc:datafield[@tag='100']/marc:subfield[@code='a']")
    author_gnd = extract_text("marc:datafield[@tag='100']/marc:subfield[@code='0']")
    year = extract_text("marc:datafield[@tag='264']/marc:subfield[@code='c']")
    links = extract_text("marc:datafield[@tag='856']/marc:subfield[@code='u']")

    return {"idn": idn, "titel": titel, "author": author, "gnd":author_gnd, "year": year, "links":links}
                                  

In [15]:
result = [parse_record(record) for record in records]
df = pd.DataFrame(result)
df

,idn,titel,author,gnd,year,links
0,1393000940,"Die Kreuzzüge neu erzählt: Religion, Land und ...","Winter, Max",None,2026,https://nbn-resolving.org/urn:nbn:de:101:1-260...
1,1392999154,"Die Kreuzzüge neu erzählt: Religion, Land und ...","Winter, Max",None,2026,https://nbn-resolving.org/urn:nbn:de:101:1-260...
2,1387832093,Die Kreuzzugsbewegung im römisch-deutschen Rei...,"Jaspert, Nikolas",(DE-588)124542115; https://d-nb.info/gnd/12454...,2026,https://doi.org/10.11588/heidok.00037971; http...
3,1388589478,Heiliger Krieg Traf auf Politische Interessen,"Weiß, Luisa",None,2026,https://nbn-resolving.org/urn:nbn:de:101:1-260...
4,1388639130,Heiliger Krieg Traf auf Politische Interessen,"Weiß, Luisa",None,2026,https://nbn-resolving.org/urn:nbn:de:101:1-260...
...,...,...,...,...,...,...
701,965428664,Geschichte der Kreuzzüge,"Henne am Rhyn, Otto",(DE-588)120474808; https://d-nb.info/gnd/12047...,None,None
702,974474908,Geschichte der Kreuzzüge,"Waas, Adolf",(DE-588)117075922; https://d-nb.info/gnd/11707...,None,https://services.dnb.de/plus/idn/974474908/blurb/
703,550622675,Geschichte der Kreuzzüge nach morgenländischen...,"Wilken, Friedrich",(DE-588)117384259; https://d-nb.info/gnd/11738...,None,None
704,550987754,Les chroniques de Jherusalem abregies,None,None,None,None


### Speichern:

In [16]:
df.to_csv("Kreuzzüge.csv", index=False, encoding="utf-8")